# EVT (padronizado)

*By Miguel Ferreira*

**Este notebook, assim com todos os outros de cada ferramenta do envelope de risco, segue o mesmo protocolo:**
1. Importação do dataset e bibliotecas
2. Execução do ```setup()``` e alinhamento temporal
3. Construção da ferramenta de risco 
4. Sanity checks mínimos
5. DataFrame final (features de risco) 
6. Padronização
7. Salvamento
Esta padronização é uma peça fundamental para um projeto **clean code.** Tanto que esta introdução estará presente em todos os notebooks de todas as ferramentas do envelope de risco.
---

In [1]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path

from scipy.stats import genpareto

In [2]:
sys.path.append(str(Path("../../src").resolve()))
from setup import setup

In [3]:
# carregar dataset base
CSV_PATH = "../../data/processed/financial_tools_datset.csv"
raw_df = pd.read_csv(CSV_PATH)

raw_df

,Date,Price,Open,High,Low,Change %,short_mavg,long_mavg,signal,EMA10,...,MOM30,RSI10,RSI30,RSI200,%K10,%D10,%K30,%D30,%K200,%D200
0,02/11/2026,1.1871,1.1895,1.1928,1.1833,-0.20%,1.187100,1.187100,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02/10/2026,1.1895,1.1913,1.1929,1.1887,-0.17%,1.188300,1.188300,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,02/09/2026,1.1915,1.1817,1.1927,1.1809,0.74%,1.189367,1.189367,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02/08/2026,1.1827,1.1814,1.1828,1.1810,0.08%,1.187700,1.187700,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02/06/2026,1.1817,1.1778,1.1827,1.1765,0.34%,1.186500,1.186500,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1563,02/17/2020,1.0834,1.0832,1.0852,1.0821,0.04%,1.086880,1.093953,0.0,1.088208,...,-0.0212,38.792415,45.322206,46.650424,20.577617,10.056914,22.987165,20.147802,14.316860,12.548450
1564,02/14/2020,1.0830,1.0841,1.0862,1.0827,-0.09%,1.084930,1.093937,0.0,1.087261,...,-0.0310,38.469796,45.217100,46.629037,23.144105,15.712119,22.520420,21.159082,14.026163,13.178295
1565,02/13/2020,1.0840,1.0874,1.0890,1.0833,-0.29%,1.083350,1.093950,0.0,1.086668,...,-0.0188,39.859143,45.543707,46.690443,47.727273,30.482998,23.687281,23.064955,14.752907,14.365310
1566,02/12/2020,1.0871,1.0916,1.0926,1.0865,-0.39%,1.083270,1.094078,0.0,1.086747,...,-0.0009,44.199076,46.565326,46.880856,63.087248,44.652875,27.304551,24.504084,17.005814,15.261628


In [4]:
# Aqui eu já aplico o método 'setup()' imediatamente
data = setup(CSV_PATH)

X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]

In [5]:
# garantir índice temporal
df = df.sort_index()
returns = df["Price"].pct_change()

NameError: name 'df' is not defined

In [ ]:
window = 100
threshold_q = 0.95

In [ ]:
def fit_evt(series, threshold_q):
    threshold = np.quantile(series, threshold_q)
    excesses = series[series > threshold] - threshold
    
    if len(excesses) < 5:
        return np.nan, np.nan
    
    c, loc, scale = genpareto.fit(excesses)
    return c, scale

In [ ]:
evt_shape = []
evt_scale = []
evt_var   = []
evt_cvar  = []

alpha = 0.99

for i in range(len(returns)):
    
    if i < window:
        evt_shape.append(np.nan)
        evt_scale.append(np.nan)
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue
    
    window_data = returns.iloc[i-window:i].dropna()
    
    if len(window_data) < window:
        evt_shape.append(np.nan)
        evt_scale.append(np.nan)
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue
    
    xi, beta = fit_evt(window_data, threshold_q)
    
    evt_shape.append(xi)
    evt_scale.append(beta)
    
    if np.isnan(xi) or np.isnan(beta):
        evt_var.append(np.nan)
        evt_cvar.append(np.nan)
        continue
    
    threshold = np.quantile(window_data, threshold_q)
    
    # EVT VaR
    var_evt = threshold + (beta / xi) * ((1 - alpha) ** (-xi) - 1)
    
    # EVT CVaR (Expected Shortfall)
    cvar_evt = (var_evt + (beta - xi * threshold)) / (1 - xi)
    
    evt_var.append(var_evt)
    evt_cvar.append(cvar_evt)

In [ ]:
evt_df_check = pd.DataFrame({
    "evt_shape": evt_shape,
    "evt_scale": evt_scale,
    "evt_var": evt_var,
    "evt_cvar": evt_cvar
}, index=df.index)

evt_df_check.describe()

In [ ]:
evt_df_check.isna().mean()

In [ ]:
evt_dataset = pd.DataFrame(index=df.index)

evt_dataset["evt_shape"] = evt_shape
evt_dataset["evt_scale"] = evt_scale
evt_dataset["evt_var"]   = evt_var
evt_dataset["evt_cvar"]  = evt_cvar

evt_dataset.tail()

In [ ]:
'''
evt_dataset.to_parquet(
    "data/processed/evt/evt_features.parquet"
)
'''